# 🛡️ Uncertainty-Aware Medical VQA — Evaluation

Evaluate the fine-tuned BLIP-2 model with **uncertainty estimation** and **abstention**.

**What this notebook does:**
1. Loads the LoRA fine-tuned BLIP-2 model
2. Generates predictions with 3 uncertainty methods:
   - **Predictive Entropy** (token-level softmax entropy)
   - **MC Dropout** (5 stochastic forward passes)
   - **Sequence Confidence** (normalized log-probability)
3. Tunes the abstention threshold (target: 80% coverage)
4. Computes safety metrics: Risk-Coverage, AUROC, ECE
5. Saves all results

**Requirements:** T4 GPU, ~30-45 min for 50 eval samples with MC Dropout

## Cell 1: Install Dependencies

In [ ]:
%pip install -q transformers accelerate peft bitsandbytes datasets Pillow tqdm pandas nltk bert-score matplotlib

## Cell 2: Configuration

In [ ]:
import os
import numpy as np

USE_DRIVE = False
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"

MODEL_NAME = "Salesforce/blip2-opt-2.7b"
LORA_CHECKPOINT = None  # Will be set below

EVAL_SAMPLES = 50
MC_DROPOUT_PASSES = 5
MAX_NEW_TOKENS = 64
TARGET_COVERAGE = 0.80
SEED = 42

PROJECT_DIR = DRIVE_PROJECT_DIR if USE_DRIVE else "/content/kvasir-vqa"
DATA_DIR = f"{PROJECT_DIR}/data"
IMAGE_DIR = f"{DATA_DIR}/images"
RESULTS_DIR = f"{PROJECT_DIR}/results"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"

# Find best LoRA checkpoint
for name in ["best_lora", "final_lora"]:
    path = f"{CHECKPOINT_DIR}/{name}"
    if os.path.exists(path):
        LORA_CHECKPOINT = path
        break

os.makedirs(f"{RESULTS_DIR}/uncertainty", exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"LoRA checkpoint: {LORA_CHECKPOINT}")
print(f"MC Dropout passes: {MC_DROPOUT_PASSES}")
print(f"Target coverage: {TARGET_COVERAGE*100:.0f}%")

## Cell 3: Load Model + Data

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load processor
processor = Blip2Processor.from_pretrained(MODEL_NAME)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# Load base model in 8-bit
print(f"Loading {MODEL_NAME} in 8-bit...")
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
base_model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load LoRA adapter
if LORA_CHECKPOINT:
    print(f"Loading LoRA from {LORA_CHECKPOINT}...")
    model = PeftModel.from_pretrained(base_model, LORA_CHECKPOINT)
else:
    print("WARNING: No LoRA checkpoint found. Using base model.")
    model = base_model

model.eval()
print("Model loaded.")

# Load test data
test_csv = Path(DATA_DIR) / "kvasir_vqa_x1_test.csv"
test_df = pd.read_csv(test_csv)
np.random.seed(SEED)
eval_subset = test_df.sample(n=min(EVAL_SAMPLES, len(test_df)), random_state=SEED)
print(f"Eval samples: {len(eval_subset)}")

## Cell 4: Metric + Uncertainty Functions

In [ ]:
import re
from collections import Counter

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor

# ---- Text normalization ----
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

# ---- VQA metrics ----
def compute_exact_match(pred, gt): return normalize_text(pred) == normalize_text(gt)

def compute_word_f1(pred, gt):
    p, g = normalize_text(pred).split(), normalize_text(gt).split()
    if not p and not g: return 1.0
    if not p or not g: return 0.0
    c = sum((Counter(p) & Counter(g)).values())
    if c == 0: return 0.0
    return 2*(c/len(p))*(c/len(g))/((c/len(p))+(c/len(g)))

def compute_bleu_scores(pred, gt):
    p, g = normalize_text(pred).split(), normalize_text(gt).split()
    if not g: return {f'bleu_{n}': (1.0 if not p else 0.0) for n in range(1,5)}
    if not p: return {f'bleu_{n}': 0.0 for n in range(1,5)}
    smooth = SmoothingFunction().method1
    return {f'bleu_{n}': sentence_bleu([g], p, weights=tuple([1.0/n]*n+[0.0]*(4-n)), smoothing_function=smooth) for n in range(1,5)}

def compute_rouge_l(pred, gt):
    p, g = normalize_text(pred).split(), normalize_text(gt).split()
    if not p and not g: return 1.0
    if not p or not g: return 0.0
    m, n = len(g), len(p)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j] = dp[i-1][j-1]+1 if g[i-1]==p[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs = dp[m][n]
    if lcs == 0: return 0.0
    return 2*(lcs/n)*(lcs/m)/((lcs/n)+(lcs/m))

def compute_meteor(pred, gt):
    p, g = normalize_text(pred).split(), normalize_text(gt).split()
    if not g: return 1.0 if not p else 0.0
    if not p: return 0.0
    return nltk_meteor([g], p)

# ---- Uncertainty: Predictive Entropy ----
def get_entropy_and_logprob(model, processor, image, question, max_tokens=64, device="cuda"):
    """Single forward pass → prediction + entropy + log-prob."""
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, dtype=torch.float16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, do_sample=False,
            output_scores=True, return_dict_in_generate=True,
        )

    prompt_len = inputs['input_ids'].shape[1]
    gen_ids = outputs.sequences[0][prompt_len:]
    prediction = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    entropies, log_probs = [], []
    for step, score in enumerate(outputs.scores):
        probs = torch.softmax(score[0], dim=-1)
        log_p = torch.log(probs.clamp(min=1e-10))
        entropies.append(-(probs * log_p).sum().item())
        if step < len(gen_ids):
            log_probs.append(log_p[gen_ids[step]].item())

    entropy_mean = float(np.mean(entropies)) if entropies else 0.0
    log_prob_mean = float(np.mean(log_probs)) if log_probs else -10.0
    confidence = float(np.exp(log_prob_mean))

    return prediction, entropy_mean, confidence

# ---- Uncertainty: MC Dropout ----
def enable_dropout(model):
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.train()

def disable_dropout(model):
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.eval()

def mc_dropout_inference(model, processor, image, question, n_passes=5, max_tokens=64, device="cuda"):
    """N stochastic passes → lexical variance as uncertainty."""
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, dtype=torch.float16)
    prompt_len = inputs['input_ids'].shape[1]

    enable_dropout(model)
    answers = []
    for _ in range(n_passes):
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        answers.append(processor.tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True).strip())
    disable_dropout(model)

    # Pairwise word F1 → variance
    pw_f1 = []
    for i in range(len(answers)):
        for j in range(i+1, len(answers)):
            pw_f1.append(compute_word_f1(answers[i], answers[j]))
    mc_uncertainty = 1.0 - (np.mean(pw_f1) if pw_f1 else 1.0)

    # Majority answer
    normed = [normalize_text(a) for a in answers]
    majority = Counter(normed).most_common(1)[0][0]
    prediction = answers[normed.index(majority)]

    return prediction, answers, float(mc_uncertainty), len(set(normed))/len(normed)

print("All functions loaded.")

## Cell 5: Run Uncertainty-Aware Inference

For each sample: greedy generation (entropy + log-prob) → MC Dropout (5 passes)

In [ ]:
from PIL import Image
from tqdm.auto import tqdm
import time

all_results = []
start = time.time()

print(f"Running uncertainty estimation on {len(eval_subset)} samples...")
print(f"  Entropy + LogProb: 1 pass per sample")
print(f"  MC Dropout: {MC_DROPOUT_PASSES} passes per sample")
print(f"  Total passes: ~{len(eval_subset) * (1 + MC_DROPOUT_PASSES)} generations\n")

for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc="Uncertainty eval"):
    img_path = Path(IMAGE_DIR) / f"{row['img_id']}.jpg"
    if not img_path.exists():
        continue

    image = Image.open(img_path).convert('RGB')
    question = str(row['question'])
    gt = str(row['answer'])
    comp = int(row.get('complexity', 1))

    # 1. Entropy + confidence (single greedy pass)
    pred_greedy, entropy, confidence = get_entropy_and_logprob(
        model, processor, image, question, MAX_NEW_TOKENS, model.device
    )

    # 2. MC Dropout (N passes)
    pred_mc, mc_answers, mc_unc, unique_ratio = mc_dropout_inference(
        model, processor, image, question, MC_DROPOUT_PASSES, MAX_NEW_TOKENS, model.device
    )

    # Final prediction: use MC majority answer
    prediction = pred_mc

    # Combined uncertainty (weighted)
    entropy_norm = min(entropy / 10.0, 1.0)
    conf_unc = 1.0 - confidence
    combined = 0.4 * entropy_norm + 0.3 * mc_unc + 0.3 * conf_unc

    # VQA metrics
    f1 = compute_word_f1(prediction, gt)
    em = int(compute_exact_match(prediction, gt))
    bl = compute_bleu_scores(prediction, gt)
    rl = compute_rouge_l(prediction, gt)
    met = compute_meteor(prediction, gt)

    all_results.append({
        'question': question, 'ground_truth': gt, 'prediction': prediction,
        'complexity': comp, 'img_id': row['img_id'],
        'mc_predictions': mc_answers,
        # VQA metrics
        'exact_match': em, 'word_f1': f1,
        'bleu_1': bl['bleu_1'], 'bleu_4': bl['bleu_4'],
        'rouge_l': rl, 'meteor': met,
        # Uncertainty scores
        'entropy': entropy, 'confidence': confidence,
        'mc_uncertainty': mc_unc, 'mc_unique_ratio': unique_ratio,
        'combined_uncertainty': combined,
    })

elapsed = time.time() - start
print(f"\nDone! {elapsed/60:.1f} min for {len(all_results)} samples")
print(f"  Average: {elapsed/len(all_results):.1f}s per sample")

## Cell 6: Uncertainty Analysis

In [ ]:
import json

# Extract arrays
f1_scores = [r['word_f1'] for r in all_results]
entropies = [r['entropy'] for r in all_results]
confidences = [r['confidence'] for r in all_results]
mc_uncs = [r['mc_uncertainty'] for r in all_results]
combined_uncs = [r['combined_uncertainty'] for r in all_results]
complexities = [r['complexity'] for r in all_results]

# Correlation: do uncertain samples actually have lower F1?
from numpy import corrcoef
print("Correlation between uncertainty and Word F1 (negative is good):")
print(f"  Entropy vs F1:      r = {corrcoef(entropies, f1_scores)[0,1]:.3f}")
print(f"  MC Dropout vs F1:   r = {corrcoef(mc_uncs, f1_scores)[0,1]:.3f}")
print(f"  1-Confidence vs F1: r = {corrcoef([1-c for c in confidences], f1_scores)[0,1]:.3f}")
print(f"  Combined vs F1:     r = {corrcoef(combined_uncs, f1_scores)[0,1]:.3f}")

# Per-complexity uncertainty
print(f"\nMean Uncertainty by Complexity:")
print(f"  {'Level':<10} {'Entropy':>8} {'MC Unc':>8} {'Confidence':>10} {'Combined':>10} {'Avg F1':>8}")
for lvl in sorted(set(complexities)):
    idx = [i for i, c in enumerate(complexities) if c == lvl]
    print(f"  Level {lvl:<5} "
          f"{np.mean([entropies[i] for i in idx]):>7.2f} "
          f"{np.mean([mc_uncs[i] for i in idx]):>7.3f} "
          f"{np.mean([confidences[i] for i in idx]):>9.3f} "
          f"{np.mean([combined_uncs[i] for i in idx]):>9.3f} "
          f"{np.mean([f1_scores[i] for i in idx])*100:>7.1f}%")

## Cell 7: Abstention Threshold Tuning

In [ ]:
def tune_threshold(unc_scores, f1_scores, target_coverage=0.80, n_steps=100):
    """Find threshold: maximize selective F1 while maintaining target coverage."""
    unc = np.array(unc_scores)
    f1 = np.array(f1_scores)
    thresholds = np.linspace(unc.min(), unc.max(), n_steps)

    best_t, best_acc, best_cov = thresholds[-1], 0, 1.0
    results = []
    for t in thresholds:
        mask = unc <= t
        cov = mask.sum() / len(unc)
        sel_f1 = f1[mask].mean() if mask.sum() > 0 else 0
        results.append((float(t), float(cov), float(sel_f1)))
        if cov >= target_coverage and sel_f1 > best_acc:
            best_t, best_acc, best_cov = t, sel_f1, cov

    return float(best_t), best_acc, best_cov, results

# Tune on combined uncertainty
optimal_t, sel_f1, sel_cov, sweep = tune_threshold(
    combined_uncs, f1_scores, TARGET_COVERAGE
)

print(f"{'='*60}")
print(f"  ABSTENTION THRESHOLD TUNING")
print(f"{'='*60}")
print(f"  Target coverage: {TARGET_COVERAGE*100:.0f}%")
print(f"  Optimal threshold: {optimal_t:.4f}")
print(f"  Coverage at threshold: {sel_cov*100:.1f}%")
print(f"  Selective F1 (answered only): {sel_f1*100:.1f}%")
print(f"  Overall F1 (all samples): {np.mean(f1_scores)*100:.1f}%")
print(f"  Improvement: +{(sel_f1 - np.mean(f1_scores))*100:.1f}%")

# Apply abstention
abstained = [i for i, u in enumerate(combined_uncs) if u > optimal_t]
answered = [i for i, u in enumerate(combined_uncs) if u <= optimal_t]
print(f"\n  Answered: {len(answered)} | Abstained: {len(abstained)} | Total: {len(all_results)}")

# Per-complexity abstention
print(f"\n  Abstention by Complexity:")
for lvl in sorted(set(complexities)):
    lvl_idx = [i for i, c in enumerate(complexities) if c == lvl]
    lvl_abs = [i for i in lvl_idx if combined_uncs[i] > optimal_t]
    print(f"    Level {lvl}: {len(lvl_abs)}/{len(lvl_idx)} abstained ({len(lvl_abs)/len(lvl_idx)*100:.0f}%)")
print(f"{'='*60}")

## Cell 8: Safety Metrics + Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

unc_arr = np.array(combined_uncs)
f1_arr = np.array(f1_scores)

# ---- AUROC ----
def compute_auroc(unc, correct):
    incorrect_idx = np.where(correct == 0)[0]
    correct_idx = np.where(correct == 1)[0]
    if len(incorrect_idx) == 0 or len(correct_idx) == 0:
        return 0.5
    concordant = sum(
        1 if unc[i] > unc[j] else 0.5 if unc[i] == unc[j] else 0
        for i in incorrect_idx for j in correct_idx
    )
    return concordant / (len(incorrect_idx) * len(correct_idx))

binary_correct = (f1_arr >= 0.5).astype(int)
auroc = compute_auroc(unc_arr, binary_correct)

# ---- Risk-Coverage ----
sorted_idx = np.argsort(unc_arr)
sorted_f1 = f1_arr[sorted_idx]
coverages = [(n+1)/len(sorted_f1) for n in range(len(sorted_f1))]
sel_accs = [sorted_f1[:n+1].mean() for n in range(len(sorted_f1))]
risks = [1 - a for a in sel_accs]
auc_risk = float(np.trapz(risks, coverages))

# ---- ECE ----
def compute_ece(conf, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0
    bin_data = []
    for i in range(n_bins):
        mask = (conf >= bins[i]) & (conf < bins[i+1]) if i < n_bins-1 else (conf >= bins[i]) & (conf <= bins[i+1])
        n = mask.sum()
        if n == 0:
            bin_data.append((0, 0, 0))
            continue
        avg_conf = conf[mask].mean()
        avg_acc = correct[mask].mean()
        ece += (n / len(conf)) * abs(avg_acc - avg_conf)
        bin_data.append((float(avg_conf), float(avg_acc), int(n)))
    return ece, bin_data

conf_arr = np.array(confidences)
ece, ece_bins = compute_ece(conf_arr, f1_arr)

# ---- Selective accuracy at key coverage levels ----
sel_acc_table = {}
for target_cov in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    n = max(1, int(target_cov * len(sorted_f1)))
    sel_acc_table[target_cov] = float(sorted_f1[:n].mean())

print(f"{'='*60}")
print(f"  SAFETY METRICS")
print(f"{'='*60}")
print(f"  AUROC:       {auroc:.3f}  {'(good)' if auroc > 0.6 else '(fair)' if auroc > 0.5 else '(poor)'}")
print(f"  AUC-Risk:    {auc_risk:.3f}  (lower is better)")
print(f"  ECE:         {ece:.3f}  (lower is better)")
print(f"\n  Selective Accuracy by Coverage:")
for cov, acc in sel_acc_table.items():
    marker = " ← target" if abs(cov - TARGET_COVERAGE) < 0.01 else ""
    print(f"    {cov*100:>5.0f}% coverage → {acc*100:.1f}% avg F1{marker}")
print(f"{'='*60}")

# ---- PLOTS ----
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Risk-Coverage curve
ax = axes[0, 0]
ax.plot(coverages, [a*100 for a in sel_accs], 'b-', linewidth=2, label='Selective F1')
ax.axhline(y=np.mean(f1_scores)*100, color='r', linestyle='--', alpha=0.7, label=f'Overall F1 ({np.mean(f1_scores)*100:.1f}%)')
ax.axvline(x=TARGET_COVERAGE, color='g', linestyle=':', alpha=0.7, label=f'Target coverage ({TARGET_COVERAGE*100:.0f}%)')
ax.set_xlabel('Coverage (fraction answered)')
ax.set_ylabel('Selective Word F1 (%)')
ax.set_title(f'Selective Accuracy vs Coverage (AUROC={auroc:.3f})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.05)

# 2. Uncertainty vs F1 scatter
ax = axes[0, 1]
colors = ['#2ecc71' if f >= 0.5 else '#e74c3c' for f in f1_scores]
ax.scatter(combined_uncs, [f*100 for f in f1_scores], c=colors, alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
ax.axvline(x=optimal_t, color='orange', linestyle='--', linewidth=2, label=f'Threshold τ={optimal_t:.3f}')
ax.set_xlabel('Combined Uncertainty')
ax.set_ylabel('Word F1 (%)')
ax.set_title('Uncertainty vs Answer Quality')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Reliability diagram (ECE)
ax = axes[1, 0]
bin_confs = [b[0] for b in ece_bins if b[2] > 0]
bin_accs = [b[1] for b in ece_bins if b[2] > 0]
bin_counts = [b[2] for b in ece_bins if b[2] > 0]
ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.7, color='#3498db', label='Actual accuracy')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.7, label='Perfect calibration')
ax.set_xlabel('Confidence')
ax.set_ylabel('Accuracy (Word F1)')
ax.set_title(f'Reliability Diagram (ECE={ece:.3f})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

# 4. Per-complexity uncertainty distribution
ax = axes[1, 1]
for lvl in sorted(set(complexities)):
    lvl_unc = [combined_uncs[i] for i, c in enumerate(complexities) if c == lvl]
    ax.hist(lvl_unc, bins=15, alpha=0.5, label=f'Level {lvl} (n={len(lvl_unc)})')
ax.axvline(x=optimal_t, color='orange', linestyle='--', linewidth=2, label=f'τ={optimal_t:.3f}')
ax.set_xlabel('Combined Uncertainty')
ax.set_ylabel('Count')
ax.set_title('Uncertainty Distribution by Complexity')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Uncertainty-Aware Medical VQA — Safety Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/uncertainty/safety_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Plots saved.")

## Cell 9: Save All Results

In [ ]:
# Summary JSON
uncertainty_summary = {
    'model': MODEL_NAME,
    'method': 'LoRA fine-tuned + uncertainty',
    'lora_checkpoint': LORA_CHECKPOINT,
    'eval_samples': len(all_results),
    'mc_dropout_passes': MC_DROPOUT_PASSES,
    'target_coverage': TARGET_COVERAGE,
    'vqa_metrics': {
        'exact_match_pct': round(np.mean([r['exact_match'] for r in all_results])*100, 2),
        'word_f1': round(np.mean(f1_scores)*100, 2),
        'bleu_1': round(np.mean([r['bleu_1'] for r in all_results])*100, 2),
        'bleu_4': round(np.mean([r['bleu_4'] for r in all_results])*100, 2),
        'rouge_l': round(np.mean([r['rouge_l'] for r in all_results])*100, 2),
        'meteor': round(np.mean([r['meteor'] for r in all_results])*100, 2),
    },
    'uncertainty': {
        'mean_entropy': round(float(np.mean(entropies)), 4),
        'mean_mc_uncertainty': round(float(np.mean(mc_uncs)), 4),
        'mean_confidence': round(float(np.mean(confidences)), 4),
        'mean_combined': round(float(np.mean(combined_uncs)), 4),
    },
    'safety_metrics': {
        'auroc': round(auroc, 4),
        'auc_risk': round(auc_risk, 4),
        'ece': round(ece, 4),
    },
    'abstention': {
        'threshold': round(optimal_t, 4),
        'coverage': round(sel_cov, 4),
        'selective_f1': round(sel_f1*100, 2),
        'overall_f1': round(np.mean(f1_scores)*100, 2),
        'improvement': round((sel_f1 - np.mean(f1_scores))*100, 2),
        'n_answered': len(answered),
        'n_abstained': len(abstained),
    },
    'selective_accuracy': {f"{int(k*100)}pct": round(v*100, 2) for k, v in sel_acc_table.items()},
}

with open(f"{RESULTS_DIR}/uncertainty/uncertainty_summary.json", 'w') as f:
    json.dump(uncertainty_summary, f, indent=2)

# Per-prediction CSV (without mc_predictions list for CSV compatibility)
csv_data = []
for r in all_results:
    row = {k: v for k, v in r.items() if k != 'mc_predictions'}
    row['abstained'] = combined_uncs[all_results.index(r)] > optimal_t
    csv_data.append(row)
pd.DataFrame(csv_data).to_csv(f"{RESULTS_DIR}/uncertainty/uncertainty_predictions.csv", index=False)

print(f"Summary  → {RESULTS_DIR}/uncertainty/uncertainty_summary.json")
print(f"CSV      → {RESULTS_DIR}/uncertainty/uncertainty_predictions.csv")
print(f"Plots    → {RESULTS_DIR}/uncertainty/safety_plots.png")

## Cell 10: Sample Predictions with Uncertainty

In [ ]:
print("\nSample Predictions with Uncertainty Scores")
print("="*90)

for i in range(min(15, len(all_results))):
    r = all_results[i]
    f1 = r['word_f1']
    unc = combined_uncs[i]
    will_abstain = unc > optimal_t

    if will_abstain:
        label = "ABSTAIN 🚫"
    elif r['exact_match']:
        label = "EXACT ✓"
    elif f1 >= 0.5:
        label = "PARTIAL ~"
    else:
        label = "WRONG ✗"

    print(f"[{i+1}] {label} | C{r['complexity']} | F1={f1:.2f} | Unc={unc:.3f} | Conf={r['confidence']:.3f}")
    print(f"  Q:    {r['question'][:85]}")
    print(f"  GT:   {r['ground_truth'][:85]}")
    print(f"  Pred: {r['prediction'][:85]}")
    if r['mc_unique_ratio'] > 0.4:
        print(f"  MC answers ({len(set([normalize_text(a) for a in r['mc_predictions']]))} unique): {[a[:40] for a in r['mc_predictions']]}")
    print("-"*90)

## Cell 11: Download Results

In [ ]:
print("Output files:")
for root, dirs, files in os.walk(f"{RESULTS_DIR}/uncertainty"):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {fpath} ({os.path.getsize(fpath)/1024:.1f} KB)")

if not USE_DRIVE:
    from google.colab import files
    try:
        files.download(f"{RESULTS_DIR}/uncertainty/uncertainty_summary.json")
        files.download(f"{RESULTS_DIR}/uncertainty/uncertainty_predictions.csv")
        files.download(f"{RESULTS_DIR}/uncertainty/safety_plots.png")
    except Exception as e:
        print(f"Download failed: {e}")